<div dir="rtl">

## بخش اول: شناخت داده‌ها و ساختار

### تسک ۱: منشأ داده و سازوکار جمع‌آوری
داده‌ها از پایگاه داده مرکز ملی هماهنگی آلزایمر آمریکا (NACC) گرفته شده است. این مرکز از سال ۲۰۰۵ اطلاعات شرکت‌کنندگان را از مراکز تحقیقاتی بیماری آلزایمر (ADRCs) در سراسر آمریکا جمع‌آوری می‌کند. 
زیرمجموعه‌ای که در این پروژه استفاده می‌کنیم شامل دو بخش اصلی است:
- **داده‌های بالینی و شناختی (UDS):** این اطلاعات در جریان ویزیت‌های سالانه شرکت‌کنندگان، از طریق معاینه مستقیم پزشک، مصاحبه با خود فرد یا همراه او، و گرفتن تست‌های استاندارد حافظه و شناخت روی فرم‌های مشخص ثبت شده است.
- **داده‌های تصویربرداری (MRI):** برای بخشی از این شرکت‌کنندگان اسکن مغزی انجام شده و اندازه‌گیری‌های عددی مربوط به حجم ماده خاکستری و ضخامت قشر نواحی مختلف مغز در این فایل قرار گرفته است.

از نظر آماری، این داده‌ها حاصل یک **مطالعه مشاهده‌ای** هستند؛ یعنی پژوهشگران هیچ مداخله یا آزمایشی روی افراد انجام نداده‌اند و صرفاً وضعیت سلامت، فشارخون، شاخص‌های بدنی و وضعیت مغزی آن‌ها را در مراجعات بالینی ثبت کرده‌اند. به همین خاطر، از این داده‌ها فقط می‌توان برای بررسی «ارتباط» بین متغیرها استفاده کرد و نمی‌توان نتیجه علت و معلولی قطعی گرفت.

در استفاده از این داده‌ها چند محدودیت مهم وجود دارد؛ اول اینکه دسترسی به پایگاه اصلی NACC عمومی نیست و در این پروژه هم ما به کل پایگاه داده دسترسی نداریم، بلکه فقط با یک زیرمجموعه از پیش تعیین‌شده کار می‌کنیم. دوم اینکه چون داده‌ها از مراکز تحقیقاتی مختلف جمع‌آوری شده‌اند، ممکن است دستگاه‌های تصویربرداری MRI یا نحوه تشخیص بالینی توسط پزشکان بین مراکز مختلف کمی متفاوت باشد.

</div>

<div dir="rtl">

### تسک ۲: واحد مشاهده و ساختار داده
در ساختار اصلی پایگاه داده NACC، اطلاعات بیماران به صورت طولی ثبت می‌شود؛ یعنی یک نفر ممکن است چندین بار به کلینیک مراجعه کرده باشد و چند ردیف داده داشته باشد. اما برای انجام تحلیل‌های آماری معتبر در این فاز، نیاز داریم که اصل استقلال مشاهدات رعایت شود.

با بررسی ابعاد داده‌ها در فایل‌های `udslong_imputed.csv` و `mriscanclean.csv` توسط پایتون، مشخص شد که داده‌هایی که برای این پروژه داریم فیلتر شده است. بنابراین، **واحد مشاهده** در داده‌های ما، دقیقاً نماینده **«یک شرکت‌کننده یکتا در یک مقطع زمانی مشخص»** است. 

همان‌طور که در خروجی کد زیر مشاهده می‌شود، هر دو فایل دارای ۱۲۳۷ سطر و ۱۲۳۷ شناسه یکتا (`NACCID`) هستند و هیچ رکورد تکراری برای بیماران در هیچ‌کدام از فایل‌ها وجود ندارد.

</div>

In [3]:
import pandas as pd

uds_df = pd.read_csv('../data/udslong_imputed.csv')
mri_df = pd.read_csv('../data/mriscanclean.csv')

print("--- بررسی مشاهدات در فایل بالینی (UDS) ---")
print(f"تعداد کل سطرها: {len(uds_df)}")
print(f"تعداد شناسه‌های یکتا (NACCID): {uds_df['NACCID'].nunique()}")

print("\n--- بررسی مشاهدات در فایل تصویربرداری (MRI) ---")
print(f"تعداد کل سطرها: {len(mri_df)}")
print(f"تعداد شناسه‌های یکتا (NACCID): {mri_df['NACCID'].nunique()}")

--- بررسی مشاهدات در فایل بالینی (UDS) ---
تعداد کل سطرها: 1237
تعداد شناسه‌های یکتا (NACCID): 1237

--- بررسی مشاهدات در فایل تصویربرداری (MRI) ---
تعداد کل سطرها: 1237
تعداد شناسه‌های یکتا (NACCID): 1237


<div dir="rtl">

### تسک ۳: نمونه‌گیری، انتخاب و قابلیت تعمیم

**سازوکار انتخاب و نمونه‌گیری:**
داده‌ها بر اساس نمونه‌گیری تصادفی از کل جامعه جمع‌آوری نشده‌اند. شرکت‌کنندگان شامل افرادی هستند که به مراکز تحقیقاتی بیماری آلزایمر (ADRCs) مراجعه کرده‌اند. این مراجعات معمولاً یا به صورت داوطلبانه (مثلاً به دلیل نگرانی فرد از وضعیت حافظه‌اش یا داشتن سابقه خانوادگی) و یا از طریق ارجاع توسط پزشکان متخصص صورت گرفته است. بنابراین، روش نمونه‌گیری از نوع **غیراحتمالی** است. 

**قابلیت تعمیم:**
از آنجا که نمونه‌گیری تصادفی نبوده و افراد از محیط‌های کلینیکی و تحقیقاتی انتخاب شده‌اند، **نمی‌توانیم نتایج پروژه را به کل جامعه سالمندان عادی تعمیم دهیم.** این نتایج صرفاً به جمعیتی با ویژگی‌های مشابه قابل تعمیم است؛ یعنی افرادی که به مراکز تحقیقاتی بیماری آلزایمر مراجعه می‌کنند و در مطالعات مرتبط با ارزیابی‌های شناختی و تصویربرداری مغز شرکت می‌کنند.

</div>

<div dir="rtl">

### تسک ۴: طبقه‌بندی متغیرها و مقیاس اندازه‌گیری

در جدول زیر، متغیرهایی که برای این پروژه انتخاب کرده‌ایم به همراه نوع آماری و نقش آن‌ها مشخص شده‌اند. برای بررسی ساختار لوب پیشانی در داده‌های MRI، تمرکز ما بر ناحیه **پیشانی فوقانی (Superior Frontal)** در هر دو نیمکره چپ و راست است که نقش کلیدی در عملکردهای شناختی و حافظه دارد.

| نام متغیر | منبع | توضیح متغیر | نوع آماری (مقیاس) | نقش در تحلیل |
| :--- | :--- | :--- | :--- | :--- |
| **`NACCID`** | هر دو فایل | شناسه یکتای بیمار | کیفی (اسمی) | کلید اتصال دو فایل |
| **`NACCBMI`** | بالینی | شاخص توده بدنی (BMI) | کمّی پیوسته (نسبی) | متغیر مستقل (ریسک‌فاکتور) |
| **`BPSYS`** | بالینی | فشار خون سیستولیک | کمّی پیوسته (نسبی) | متغیر مستقل (ریسک‌فاکتور) |
| **`HYPERTEN`** | بالینی | سابقه پرفشاری خون (کد ۰، ۱، ۲) | کیفی (اسمی) | متغیر مستقل (ریسک‌فاکتور) |
| **`MOCATOTS`** | بالینی | نمره تست شناختی MoCA (۰ تا ۳۰) | کمّی گسسته (نسبی) | متغیر وابسته (پیامد شناختی) |
| **`LH_SUPERIORFRONTAL_GVOL`** | MRI | حجم ماده خاکستری پیشانی فوقانی (چپ) | کمّی پیوسته (نسبی) | متغیر وابسته (ساختار مغز) |
| **`RH_SUPERIORFRONTAL_GVOL`** | MRI | حجم ماده خاکستری پیشانی فوقانی (راست) | کمّی پیوسته (نسبی) | متغیر وابسته (ساختار مغز) |
| **`LH_SUPERIORFRONTAL_AVGTH`** | MRI | ضخامت قشر پیشانی فوقانی (چپ) | کمّی پیوسته (نسبی) | متغیر وابسته (ساختار مغز) |
| **`RH_SUPERIORFRONTAL_AVGTH`** | MRI | ضخامت قشر پیشانی فوقانی (راست) | کمّی پیوسته (نسبی) | متغیر وابسته (ساختار مغز) |
| **`NACCAGE`** | بالینی | سن بیمار (سال) | کمّی پیوسته (نسبی) | متغیر کنترل (زمینه) |
| **`SEX`** | بالینی | جنسیت (۱=مرد، ۲=زن) | کیفی (اسمی) | متغیر کنترل (زمینه) |
| **`EDUC`** | بالینی | سنوات تحصیلات (سال) | کمّی گسسته (نسبی) | متغیر کنترل (زمینه) |

**نکته:** متغیرهای کیفی مانند جنسیت و سابقه فشار خون در فایل داده با عدد کدگذاری شده‌اند، اما ماهیت آماری آن‌ها دسته‌بندی است و در تحلیل‌های بعدی به همین صورت با آن‌ها رفتار خواهد شد.

</div>

In [5]:
selected_uds = ['NACCID', 'NACCAGE', 'SEX', 'EDUC', 'NACCBMI', 'BPSYS', 'HYPERTEN', 'MOCATOTS']
selected_mri = ['NACCID', 'LH_SUPERIORFRONTAL_GVOL', 'RH_SUPERIORFRONTAL_GVOL', 
                'LH_SUPERIORFRONTAL_AVGTH', 'RH_SUPERIORFRONTAL_AVGTH']

print("--- تعداد مقادیر گمشده در داده‌های بالینی (UDS) ---")
print(uds_df[selected_uds].isna().sum())

print("\n--- تعداد مقادیر گمشده در داده‌های تصویربرداری (MRI) ---")
print(mri_df[selected_mri].isna().sum())

print("\n--- خلاصه وضعیت داده‌های بالینی ---")
display(uds_df[selected_uds].describe().round(2))

print("\n--- خلاصه وضعیت داده‌های تصویربرداری ---")
display(mri_df[selected_mri].describe().round(2))

--- تعداد مقادیر گمشده در داده‌های بالینی (UDS) ---
NACCID      0
NACCAGE     0
SEX         0
EDUC        0
NACCBMI     0
BPSYS       0
HYPERTEN    0
MOCATOTS    0
dtype: int64

--- تعداد مقادیر گمشده در داده‌های تصویربرداری (MRI) ---
NACCID                      0
LH_SUPERIORFRONTAL_GVOL     0
RH_SUPERIORFRONTAL_GVOL     0
LH_SUPERIORFRONTAL_AVGTH    0
RH_SUPERIORFRONTAL_AVGTH    0
dtype: int64

--- خلاصه وضعیت داده‌های بالینی ---


,NACCAGE,SEX,EDUC,NACCBMI,BPSYS,HYPERTEN,MOCATOTS
count,1237.00,1237.00,1237.00,1237.00,1237.00,1237.00,1237.00
mean,68.19,1.60,16.39,117.03,214.11,0.50,24.65
std,8.04,0.49,4.29,261.98,221.36,0.97,5.75
min,41.00,1.00,0.00,15.60,83.00,0.00,0.00
25%,63.00,1.00,14.00,24.10,122.00,0.00,23.00
50%,68.00,2.00,16.00,27.30,135.00,0.00,25.40
75%,74.00,2.00,18.00,32.20,152.00,1.00,27.00
max,91.00,2.00,99.00,888.80,888.00,9.00,88.00



--- خلاصه وضعیت داده‌های تصویربرداری ---


,LH_SUPERIORFRONTAL_GVOL,RH_SUPERIORFRONTAL_GVOL,LH_SUPERIORFRONTAL_AVGTH,RH_SUPERIORFRONTAL_AVGTH
count,1237.00,1237.00,1237.00,1237.00
mean,21764.81,23988.37,2.49,2.47
std,3145.38,3635.76,0.15,0.14
min,2035.00,2354.00,1.03,1.12
25%,19963.00,21846.00,2.42,2.40
50%,21706.00,23822.00,2.50,2.48
75%,23571.00,26158.00,2.58,2.56
max,31261.00,43087.00,2.90,2.91


<div dir="rtl">

**نتیجه‌گیری بررسی کیفیت داده‌ها:**
همان‌طور که در خروجی‌های بالا مشاهده می‌شود، هیچ مقدار گمشده استانداردی (`NaN`) در داده‌ها وجود ندارد. با این حال، بررسی جدول آمار توصیفی در داده‌های بالینی (UDS) نشان می‌دهد که از یک‌سری کدهای قراردادی عددی (اعداد بسیار بزرگ یا خارج از محدوده منطقی) به جای مقادیر گمشده استفاده شده است. این اعداد در واقع کدهای تعریف‌شده توسط NACC برای داده‌های «نامشخص»، «اخذنشده» یا «امتناع از پاسخ» هستند. از سوی دیگر، داده‌های تصویربرداری (MRI) تمیز بوده و فاقد مقادیر پرت یا کدهای نامعتبر هستند.

</div>